In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import time
import pandas as pd

In [36]:
MAIN_URL = 'https://fri.uni-lj.si/sl/o-fakulteti/osebje'

driver = webdriver.Chrome()
driver.get(MAIN_URL)

listed_by_letter = driver.find_elements(By.CLASS_NAME, 'listedProf')

In [37]:
for list in listed_by_letter:
    print(list.text)
    print('------------------')

Marija Absec
Areeb Ahmed
Špela Antloga
Tjaša Arčon
Špela Arhar Holdt
Nina Arko Krmelj
Fatemeh Azad
------------------
Marko Bajec
Borut Batagelj
Andrej Bauer
Miha Bejek
Simon Bele
Aljoša Besednjak
Luka Bezgovšek
Marko Boben
Ciril Bohak
Alenka Bone
Tadej Borovšak
Zoran Bosnić
Narvika Bovcon
Ivan Bratko
Marko Brodarič
Andrej Brodnik
Patricio Bulić
------------------
Tilen Celjer
Mojca Ciglarič
Dekan
Zala Cimperman
Tomaž Curk
Prodekan
Jernej Cvek
------------------
Luka Čehovin Zajc
Rok Češnovar
Jaka Čibej
Uroš Čibej
Božica Čulibrk Rajkovič
------------------
Andrea De Gobbis
Harun Demir
Janez Demšar
Jure Demšar
Tomaž Dobravec
Matej Dobrevski
------------------
Žiga Emeršič
Dániel Erdélyhegyi
Aleš Erjavec
------------------
Jana Faganeli Pucer
Gašper Fele Žorž
Gašper Fijavž
Marko Firm
Borut Flis
Aleksandra Franc
Damir Franetič
Matic Fučka
Damjan Fujs
Svetlana Fuks
Erika Funa
Luka Fürst
------------------
Gregor Gabrovšek
Peter Gabrovšek
Timotej Gale
Kaja Gantar
Apolonija Gantar
Bruno Gašp

In [25]:
for list in listed_by_letter:
    for prof in list.find_elements(By.TAG_NAME, 'a'):
        print(prof.text)
        print('------------------')

Marija Absec
------------------
Areeb Ahmed
------------------
Špela Antloga
------------------
Tjaša Arčon
------------------
Špela Arhar Holdt
------------------
Nina Arko Krmelj
------------------
Fatemeh Azad
------------------
Marko Bajec
------------------
Borut Batagelj
------------------
Andrej Bauer
------------------
Miha Bejek
------------------
Simon Bele
------------------
Aljoša Besednjak
------------------
Luka Bezgovšek
------------------
Marko Boben
------------------
Ciril Bohak
------------------
Alenka Bone
------------------
Tadej Borovšak
------------------
Zoran Bosnić
------------------
Narvika Bovcon
------------------
Ivan Bratko
------------------
Marko Brodarič
------------------
Andrej Brodnik
------------------
Patricio Bulić
------------------
Tilen Celjer
------------------
Mojca Ciglarič
Dekan
------------------
Zala Cimperman
------------------
Tomaž Curk
Prodekan
------------------
Jernej Cvek
------------------
Luka Čehovin Zajc
------------------
Ro

In [17]:
#WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Soglašam')]"))).click()

In [64]:
df = pd.DataFrame(columns=['Name', 'Title', 'Number', 'Number of courses', 'Number of projects'])
df

,Name,Title,Number,Number of courses,Number of projects


In [38]:
profile_urls = []
for list in listed_by_letter:
    prof_links = list.find_elements(By.TAG_NAME, 'a')
    for prof in prof_links:
        profile_urls.append(prof.get_attribute('href'))  # Store the href of each link

In [69]:
len(profile_urls)

293

In [76]:
for url in profile_urls[11:]:
    driver.get(url)
    time.sleep(2) 

    # Wait for the 'profName' element to be visible, up to 10 seconds
    prof_name = WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.ID, 'profName'))
    ).text

     # Similarly wait for 'profTitle' and 'telephoneInfo' elements
    prof_title = WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.ID, 'profTitle'))
    ).text
    
    try:
        prof_number_element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, 'telephoneInfo'))
        )
        prof_number = prof_number_element.text if prof_number_element else ""
    except TimeoutException:
        prof_number = ""
        
    # Courses
    num_courses = 0
    headers = driver.find_elements(By.CLASS_NAME, "aboutMainSectionContainer")
    for header in headers:
        # print(header.text)
        if "PREDMETI" in header.text:
            courses_div = header.find_element(By.CLASS_NAME, "aboutMainSectionLi")
            courses_list = courses_div.find_element(By.TAG_NAME, "ul")
            course_items = courses_list.find_elements(By.TAG_NAME, "li")
            num_courses = len(course_items)
            break
            
    # Projects
    num_active_projects = 0
    num_completed_projects = 0
    for header in headers:
        if "PROJEKTI" in header.text:
            projects_div = header.find_element(By.CLASS_NAME, "aboutMainSectionLi")
            projects_list = projects_div.find_elements(By.TAG_NAME, "ul")
            # print(projects_list)
            project_items = projects_list[0].find_elements(By.TAG_NAME, "li")
            num_active_projects = len(project_items)
            if len(projects_list) > 1:
                completed_project_items = projects_list[1].find_elements(By.TAG_NAME, "li")
                num_completed_projects = len(completed_project_items)
            
    total_projects = num_active_projects + num_completed_projects

    length = len(df)
    df.loc[length] = {
        'Name': prof_name,
        'Title': prof_title,
        'Number': prof_number,
        'Number of courses': num_courses,
        'Number of projects': total_projects
    }
    
    print(f"{prof_name} - {prof_title} - {prof_number} - Courses: {num_courses} - Projects: {total_projects}")

Simon Bele - Tehniški sodelavec -  - Courses: 0 - Projects: 1
Aljoša Besednjak - Vzdrževalec - T: +386 41 304 743 - Courses: 0 - Projects: 0
Luka Bezgovšek - Tehniški sodelavec -  - Courses: 0 - Projects: 0
asist. dr. Marko Boben - Asistent - T: +386 1 479 8258 - Courses: 1 - Projects: 1
doc. dr. Ciril Bohak - Docent - T: +386 1 479 8259 - Courses: 4 - Projects: 15
Alenka Bone - Kadrovska služba - T: +386 1 4798115 - Courses: 0 - Projects: 0
Tadej Borovšak - Drugi člani -  - Courses: 0 - Projects: 0
prof. dr. Zoran Bosnić - Redni profesor - T: +386 1 479 8237 - Courses: 18 - Projects: 21
prof. dr. Narvika Bovcon - Redni profesor - T: +386 1 479 8272 - Courses: 5 - Projects: 5
prof. dr. Ivan Bratko - Raziskovalec - T: +386 1 479 8287 - Courses: 4 - Projects: 12
Marko Brodarič - Raziskovalec -  - Courses: 0 - Projects: 1
doc. dr. Andrej Brodnik - Docent - T: +386 1 479 8263 - Courses: 3 - Projects: 26
prof. dr. Patricio Bulić - Redni profesor - T: +386 1 479 8255 - Courses: 3 - Projects:

In [77]:
df

,Name,Title,Number,Number of courses,Number of projects
0,Marija Absec,Tehniški sodelavec,,0,0
1,dr. Areeb Ahmed,Drugi člani,,0,0
2,Špela Antloga,Raziskovalec,,0,0
3,Tjaša Arčon,Raziskovalec,,0,0
4,znan. sod. dr. Špela Arhar Holdt,Drugi člani,,0,4
...,...,...,...,...,...
288,izr. prof. dr. Slavko Žitnik,Izredni profesor\nProdekan,T: +386 1 476 82 16,13,15
289,Urška Žnidarič,Drugi člani,T: +386 1 4798111,0,0
290,Klara Žnideršič,Tehniški sodelavec,,0,1
291,doc. dr. Bojan Žunkovič,Asistent,,6,1


In [78]:
df.to_csv('data/professors_data.csv', index=False)

In [79]:
driver.quit()

In [62]:
for url in profile_urls[7:9]:
    driver.get(url)
    time.sleep(5)
    
    num_courses = 0
    headers = driver.find_elements(By.CLASS_NAME, "aboutMainSectionContainer")
    for header in headers:
        # print(header.text)
        if "PREDMETI" in header.text:
            courses_div = header.find_element(By.CLASS_NAME, "aboutMainSectionLi")
            courses_list = courses_div.find_element(By.TAG_NAME, "ul")
            course_items = courses_list.find_elements(By.TAG_NAME, "li")
            num_courses = len(course_items)
            break
    # 
    # print(f"Number of courses: {num_courses}")
 
    # # Counting projects under "Projekti" and "Zakljuceni projekti"
    num_active_projects = 0
    num_completed_projects = 0
    for header in headers:
        if "PROJEKTI" in header.text:
            projects_div = header.find_element(By.CLASS_NAME, "aboutMainSectionLi")
            projects_list = projects_div.find_elements(By.TAG_NAME, "ul")
            # print(projects_list)
            project_items = projects_list[0].find_elements(By.TAG_NAME, "li")
            num_active_projects = len(project_items)
            completed_project_items = projects_list[1].find_elements(By.TAG_NAME, "li")
            num_completed_projects = len(completed_project_items)
        # elif "Zaključeni projekti" in header.text:
        #     completed_projects_list = header.find_element(By.XPATH, "../following-sibling::ul[@class='customList']")
        #     completed_project_items = completed_projects_list.find_elements(By.TAG_NAME, "li")
        #     num_completed_projects = len(completed_project_items)

    total_projects = num_active_projects + num_completed_projects
    
    # Save data for each professor (you can also print or log it here)
    print(f"Courses: {num_courses}, Active Projects: {num_active_projects}, Completed Projects: {num_completed_projects}, Total Projects: {total_projects}")
    

Courses: 3, Active Projects: 4, Completed Projects: 33, Total Projects: 37
Courses: 3, Active Projects: 2, Completed Projects: 6, Total Projects: 8
